# colab link

https://colab.research.google.com/drive/1s01NmUW-RuZWBbEcG6CGz3htzDnPWdvN?usp=sharing

# imports and installs

In [1]:
import pandas as pd
from IPython.display import display
import duckdb

# encode traffic dataset

In [2]:
df_traffic = pd.read_csv("/content/MTA_Congestion_Relief_Zone_Vehicle_Entries__Beginning_2025_20251011.csv")
display(df_traffic.head())

,Toll Date,Toll Hour,Toll 10 Minute Block,Minute of Hour,Hour of Day,Day of Week Int,Day of Week,Toll Week,Time Period,Vehicle Class,Detection Group,Detection Region,CRZ Entries,Excluded Roadway Entries
0,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,Saturday,09/28/2025,Overnight,"1 - Cars, Pickups and Vans",Queens Midtown Tunnel,Queens,129,0
1,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,Saturday,09/28/2025,Overnight,4 - Buses,Holland Tunnel,New Jersey,2,0
2,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,Saturday,09/28/2025,Overnight,4 - Buses,FDR Drive at 60th St,FDR Drive,1,1
3,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,Saturday,09/28/2025,Overnight,4 - Buses,East 60th St,East 60th St,4,0
4,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,Saturday,09/28/2025,Overnight,4 - Buses,Williamsburg Bridge,Brooklyn,2,0


In [3]:
df_traffic_encoded = pd.get_dummies(df_traffic, columns=["Time Period","Day of Week", "Vehicle Class", "Detection Region"], drop_first=False)

display(df_traffic_encoded.head())
print(df_traffic_encoded.shape)
print(df_traffic_encoded.columns)

,Toll Date,Toll Hour,Toll 10 Minute Block,Minute of Hour,Hour of Day,Day of Week Int,Toll Week,Detection Group,CRZ Entries,Excluded Roadway Entries,...,Vehicle Class_4 - Buses,Vehicle Class_5 - Motorcycles,Vehicle Class_TLC Taxi/FHV,Detection Region_Brooklyn,Detection Region_East 60th St,Detection Region_FDR Drive,Detection Region_New Jersey,Detection Region_Queens,Detection Region_West 60th St,Detection Region_West Side Highway
0,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,09/28/2025,Queens Midtown Tunnel,129,0,...,False,False,False,False,False,False,False,True,False,False
1,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,09/28/2025,Holland Tunnel,2,0,...,True,False,False,False,False,False,True,False,False,False
2,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,09/28/2025,FDR Drive at 60th St,1,1,...,True,False,False,False,False,True,False,False,False,False
3,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,09/28/2025,East 60th St,4,0,...,True,False,False,False,True,False,False,False,False,False
4,10/04/2025,10/04/2025 11:00:00 PM,10/04/2025 11:50:00 PM,50,23,7,09/28/2025,Williamsburg Bridge,2,0,...,True,False,False,True,False,False,False,False,False,False


(2830464, 32)
Index(['Toll Date', 'Toll Hour', 'Toll 10 Minute Block', 'Minute of Hour',
       'Hour of Day', 'Day of Week Int', 'Toll Week', 'Detection Group',
       'CRZ Entries', 'Excluded Roadway Entries', 'Time Period_Overnight',
       'Time Period_Peak', 'Day of Week_Friday', 'Day of Week_Monday',
       'Day of Week_Saturday', 'Day of Week_Sunday', 'Day of Week_Thursday',
       'Day of Week_Tuesday', 'Day of Week_Wednesday',
       'Vehicle Class_1 - Cars, Pickups and Vans',
       'Vehicle Class_2 - Single-Unit Trucks',
       'Vehicle Class_3 - Multi-Unit Trucks', 'Vehicle Class_4 - Buses',
       'Vehicle Class_5 - Motorcycles', 'Vehicle Class_TLC Taxi/FHV',
       'Detection Region_Brooklyn', 'Detection Region_East 60th St',
       'Detection Region_FDR Drive', 'Detection Region_New Jersey',
       'Detection Region_Queens', 'Detection Region_West 60th St',
       'Detection Region_West Side Highway'],
      dtype='object')


In [6]:
df_traffic_encoded.to_csv("MTA_Congestion_Relief_Zone_Vehicle_Entries_Encoded.csv", index = False)

# join with subway trips

In [5]:
con = duckdb.connect(database=':memory:')

In [7]:
con.execute(
    """
    CREATE TABLE IF NOT EXISTS traffic
    AS SELECT *
    REPLACE(strptime("Toll Date", '%m/%d/%Y') AS "Toll Date")
    FROM read_csv('/content/MTA_Congestion_Relief_Zone_Vehicle_Entries_Encoded.csv', types={"Toll Date": "VARCHAR"})
    """
)

con.execute("""SELECT * FROM traffic""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Toll Date,Toll Hour,Toll 10 Minute Block,Minute of Hour,Hour of Day,Day of Week Int,Toll Week,Detection Group,CRZ Entries,Excluded Roadway Entries,...,Vehicle Class_4 - Buses,Vehicle Class_5 - Motorcycles,Vehicle Class_TLC Taxi/FHV,Detection Region_Brooklyn,Detection Region_East 60th St,Detection Region_FDR Drive,Detection Region_New Jersey,Detection Region_Queens,Detection Region_West 60th St,Detection Region_West Side Highway
0,2025-10-04,2025-10-04 23:00:00,2025-10-04 23:50:00,50,23,7,09/28/2025,Queens Midtown Tunnel,129,0,...,False,False,False,False,False,False,False,True,False,False
1,2025-10-04,2025-10-04 23:00:00,2025-10-04 23:50:00,50,23,7,09/28/2025,Holland Tunnel,2,0,...,True,False,False,False,False,False,True,False,False,False
2,2025-10-04,2025-10-04 23:00:00,2025-10-04 23:50:00,50,23,7,09/28/2025,FDR Drive at 60th St,1,1,...,True,False,False,False,False,True,False,False,False,False
3,2025-10-04,2025-10-04 23:00:00,2025-10-04 23:50:00,50,23,7,09/28/2025,East 60th St,4,0,...,True,False,False,False,True,False,False,False,False,False
4,2025-10-04,2025-10-04 23:00:00,2025-10-04 23:50:00,50,23,7,09/28/2025,Williamsburg Bridge,2,0,...,True,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830459,2025-01-05,2025-01-05 00:00:00,2025-01-05 00:00:00,0,0,1,01/05/2025,West Side Highway at 60th St,85,4,...,False,False,False,False,False,False,False,False,False,True
2830460,2025-01-05,2025-01-05 00:00:00,2025-01-05 00:00:00,0,0,1,01/05/2025,West 60th St,60,0,...,False,False,False,False,False,False,False,False,True,False
2830461,2025-01-05,2025-01-05 00:00:00,2025-01-05 00:00:00,0,0,1,01/05/2025,Queensboro Bridge,76,0,...,False,False,False,False,False,False,False,True,False,False
2830462,2025-01-05,2025-01-05 00:00:00,2025-01-05 00:00:00,0,0,1,01/05/2025,Queens Midtown Tunnel,63,0,...,False,False,False,False,False,False,False,True,False,False


In [8]:
con.execute("""DESCRIBE traffic""").df()

,column_name,column_type,null,key,default,extra
0,Toll Date,TIMESTAMP,YES,None,None,None
1,Toll Hour,TIMESTAMP,YES,None,None,None
2,Toll 10 Minute Block,TIMESTAMP,YES,None,None,None
3,Minute of Hour,BIGINT,YES,None,None,None
4,Hour of Day,BIGINT,YES,None,None,None
5,Day of Week Int,BIGINT,YES,None,None,None
6,Toll Week,VARCHAR,YES,None,None,None
7,Detection Group,VARCHAR,YES,None,None,None
8,CRZ Entries,BIGINT,YES,None,None,None
9,Excluded Roadway Entries,BIGINT,YES,None,None,None


In [9]:
con.execute(
    """
    DROP TABLE IF EXISTS traffic_grouped;
    CREATE TABLE IF NOT EXISTS traffic_grouped AS
    SELECT
        "Toll 10 Minute Block" as interval,
        sum("CRZ Entries") as entries,
        sum("Excluded Roadway Entries") as excluded_roadway_entries,
        "Time Period_Overnight" as overnight,
        "Time Period_Peak" as peak,
        "Day of Week_Monday" as monday,
        "Day of Week_Tuesday" as tuesday,
        "Day of Week_Wednesday" as wednesday,
        "Day of Week_Thursday" as thursday,
        "Day of Week_Friday" as friday,
        "Day of Week_Saturday" as saturday,
        "Day of Week_Sunday" as sunday,
        "Vehicle Class_1 - Cars, Pickups and Vans" as car_pickup_van,
        "Vehicle Class_2 - Single-Unit Trucks" as single_unit_truck,
        "Vehicle Class_3 - Multi-Unit Trucks" as multi_unit_truck,
        "Vehicle Class_4 - Buses" as bus,
        "Vehicle Class_5 - Motorcycles" as motorcycle,
        "Vehicle Class_TLC Taxi/FHV" as taxi,
        "Detection Region_Brooklyn" as brooklyn,
        "Detection Region_East 60th St" as east_60,
        "Detection Region_FDR Drive" as fdr,
        "Detection Region_New Jersey" as nj,
        "Detection Region_Queens" as queens,
        "Detection Region_West 60th St" as west_60,
        "Detection Region_West Side Highway" as west_side_hwy
    FROM traffic
    GROUP BY
        interval,
        overnight,
        peak,
        monday,
        tuesday,
        wednesday,
        thursday,
        friday,
        saturday,
        sunday,
        car_pickup_van,
        single_unit_truck,
        multi_unit_truck,
        bus,
        motorcycle,
        taxi,
        brooklyn,
        east_60,
        fdr,
        nj,
        queens,
        west_60,
        west_side_hwy
        ORDER BY
        interval,
        entries
    """
)

con.execute("select * from traffic_grouped").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,interval,entries,excluded_roadway_entries,overnight,peak,monday,tuesday,wednesday,thursday,friday,...,bus,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy
0,2025-01-05 00:00:00,0.0,0.0,True,False,False,False,False,False,False,...,False,True,False,True,False,False,False,False,False,False
1,2025-01-05 00:00:00,0.0,0.0,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,2025-01-05 00:00:00,0.0,0.0,True,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,True
3,2025-01-05 00:00:00,0.0,0.0,True,False,False,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
4,2025-01-05 00:00:00,0.0,0.0,True,False,False,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1651099,2025-10-04 23:50:00,277.0,53.0,True,False,False,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
1651100,2025-10-04 23:50:00,343.0,0.0,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1651101,2025-10-04 23:50:00,357.0,104.0,True,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
1651102,2025-10-04 23:50:00,414.0,0.0,True,False,False,False,False,False,False,...,False,False,True,False,True,False,False,False,False,False


In [10]:
con.execute("""DESCRIBE traffic_grouped""").df()

,column_name,column_type,null,key,default,extra
0,interval,TIMESTAMP,YES,None,None,None
1,entries,HUGEINT,YES,None,None,None
2,excluded_roadway_entries,HUGEINT,YES,None,None,None
3,overnight,BOOLEAN,YES,None,None,None
4,peak,BOOLEAN,YES,None,None,None
5,monday,BOOLEAN,YES,None,None,None
6,tuesday,BOOLEAN,YES,None,None,None
7,wednesday,BOOLEAN,YES,None,None,None
8,thursday,BOOLEAN,YES,None,None,None
9,friday,BOOLEAN,YES,None,None,None


In [11]:
con.execute(
    """
    CREATE TABLE IF NOT EXISTS trips
    AS SELECT * FROM read_parquet('/content/updated_trips.parquet')
    """
)

con.execute(
    """
    ALTER TABLE trips
        ALTER start TYPE TIMESTAMP;
    ALTER TABLE trips
        ALTER start_interval TYPE TIMESTAMP;
    ALTER TABLE trips
        ALTER last_seen TYPE TIMESTAMP;
    ALTER TABLE trips
        ALTER last_seen_interval TYPE TIMESTAMP;
    ALTER TABLE trips
        ALTER marked_past_time TYPE TIMESTAMP;
    ALTER TABLE trips
        ALTER marked_past_interval TYPE TIMESTAMP
    """
)

con.execute("SELECT * from trips").df()

,trip_uid,trip_id,route_id,direction_id,start,start_interval,vehicle_id,last_seen,last_seen_interval,marked_past_time,marked_past_interval,num_updates,num_schedule_changes,num_schedule_rewrites
0,1736053320_5..S32R,030200_5..S32R,5,1,2025-01-05 00:02:00,2025-01-05 00:00:00,05 0502 DYR/180,2025-01-05 05:10:03,2025-01-05 05:10:00,2025-01-05 05:10:08,2025-01-05 05:10:00,352,0,0
1,1736053350_2..S08R,030250_2..S08R,2,1,2025-01-05 00:02:30,2025-01-05 00:00:00,02 0502+ 241/FLA,2025-01-05 06:47:23,2025-01-05 06:40:00,2025-01-05 06:47:38,2025-01-05 06:40:00,924,0,0
2,1736053380_3..S01R,030300_3..S01R,3,1,2025-01-05 00:03:00,2025-01-05 00:00:00,03 0503 148/NLT,2025-01-05 06:13:23,2025-01-05 06:10:00,2025-01-05 06:13:38,2025-01-05 06:10:00,788,0,0
3,1736053380_5..N32R,030300_5..N32R,5,0,2025-01-05 00:03:00,2025-01-05 00:00:00,05 0503 180/DYR,2025-01-05 05:17:03,2025-01-05 05:10:00,2025-01-05 05:17:08,2025-01-05 05:10:00,368,0,0
4,1736053410_GS.S01R,030350_GS.S01R,GS,1,2025-01-05 00:03:30,2025-01-05 00:00:00,0S 0503+ TSS/GCS,2025-01-05 05:04:45,2025-01-05 05:00:00,2025-01-05 05:04:53,2025-01-05 05:00:00,270,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2099564,1760150160_SI..S,015600_SI..S,SI,1,2025-10-10 22:36:00,2025-10-10 22:30:00,1SI 0236 STG/TNV,2025-10-11 03:18:11,2025-10-11 03:10:00,2025-10-11 03:18:26,2025-10-11 03:10:00,668,0,0
2099565,1760151660_SI..N,018100_SI..N,SI,0,2025-10-10 23:01:00,2025-10-10 23:00:00,1SI 0301 TNV/STG,2025-10-11 03:43:13,2025-10-11 03:40:00,2025-10-11 03:43:26,2025-10-11 03:40:00,674,0,0
2099566,1760151960_SI..S,018600_SI..S,SI,1,2025-10-10 23:06:00,2025-10-10 23:00:00,1SI 0306 STG/TNV,2025-10-11 03:48:28,2025-10-11 03:40:00,2025-10-11 03:48:41,2025-10-11 03:40:00,696,0,0
2099567,1760153460_SI..N,021100_SI..N,SI,0,2025-10-10 23:31:00,2025-10-10 23:30:00,1SI 0331 TNV/STG,2025-10-11 04:12:59,2025-10-11 04:10:00,2025-10-11 04:13:13,2025-10-11 04:10:00,755,0,0


In [12]:
con.execute("""DESCRIBE trips""").df()

,column_name,column_type,null,key,default,extra
0,trip_uid,VARCHAR,YES,None,None,None
1,trip_id,VARCHAR,YES,None,None,None
2,route_id,VARCHAR,YES,None,None,None
3,direction_id,BIGINT,YES,None,None,None
4,start,TIMESTAMP,YES,None,None,None
5,start_interval,TIMESTAMP,YES,None,None,None
6,vehicle_id,VARCHAR,YES,None,None,None
7,last_seen,TIMESTAMP,YES,None,None,None
8,last_seen_interval,TIMESTAMP,YES,None,None,None
9,marked_past_time,TIMESTAMP,YES,None,None,None


In [13]:
con.execute(
    """
    DROP TABLE IF EXISTS route_avgs;
    CREATE TABLE IF NOT EXISTS route_avgs AS
    SELECT
        start_interval::TIME as start_interval,
        route_id,
        direction_id,
        avg(last_seen::time) as avg_last_seen
    FROM trips
    GROUP BY
        start_interval::TIME,
        route_id,
        direction_id,
    ORDER BY
        start_interval;
    """
)

con.execute("""SELECT * FROM route_avgs""").df()

,start_interval,route_id,direction_id,avg_last_seen
0,00:00:00,SI,1,05:07:37.610108
1,00:00:00,4,0,05:39:35.200000
2,00:00:00,GS,1,04:32:23.828571
3,00:00:00,3,0,04:43:51.563981
4,00:00:00,6,1,05:12:35.327586
...,...,...,...,...
7439,23:50:00,M,0,04:08:12.500000
7440,23:50:00,SI,1,04:06:46
7441,23:50:00,A,0,04:16:11
7442,23:50:00,F,0,04:18:36.800000


In [14]:
con.execute(
    """
    DROP TABLE IF EXISTS delays;
    CREATE TABLE IF NOT EXISTS delays AS
    SELECT
        trips.start_interval AS start_interval,
        trips.route_id,
        trips.direction_id,
        count(*) AS num_trips,
        round(avg(abs(date_diff('minute', last_seen::TIME, avg_last_seen))), 3) AS avg_diff_from_last_seen
    FROM trips
    JOIN route_avgs
        ON trips.route_id = route_avgs.route_id
        AND trips.direction_id = route_avgs.direction_id
        AND trips.start_interval::time = route_avgs.start_interval
    GROUP BY
        trips.start_interval,
        trips.direction_id,
        trips.route_id,
    ORDER BY
        start_interval,
        avg_diff_from_last_seen DESC,
        trips.route_id,
        trips.direction_id
    """
)

con.execute("SELECT * FROM delays").df()

,start_interval,route_id,direction_id,num_trips,avg_diff_from_last_seen
0,2025-01-05 00:00:00,H,0,1,119.0
1,2025-01-05 00:00:00,3,0,1,92.0
2,2025-01-05 00:00:00,A,0,1,84.0
3,2025-01-05 00:00:00,3,1,1,77.0
4,2025-01-05 00:00:00,H,1,1,65.0
...,...,...,...,...,...
1505411,2025-10-10 23:50:00,L,1,1,6.0
1505412,2025-10-10 23:50:00,E,1,1,5.0
1505413,2025-10-10 23:50:00,J,0,1,4.0
1505414,2025-10-11 00:00:00,A,0,1,39.0


In [15]:
con.execute("""DESCRIBE delays""").df()

,column_name,column_type,null,key,default,extra
0,start_interval,TIMESTAMP,YES,None,None,None
1,route_id,VARCHAR,YES,None,None,None
2,direction_id,BIGINT,YES,None,None,None
3,num_trips,BIGINT,YES,None,None,None
4,avg_diff_from_last_seen,DOUBLE,YES,None,None,None


In [16]:
con.execute(
    """
    DROP TABLE IF EXISTS delays_recent;
    CREATE TABLE delays_recent AS
    SELECT *
    FROM delays
    WHERE start_interval BETWEEN '2025-08-01' AND '2025-10-11 23:59:59';
    """
)

con.execute("SELECT * FROM delays_recent").df()

,start_interval,route_id,direction_id,num_trips,avg_diff_from_last_seen
0,2025-08-01 00:00:00,H,0,1,58.000
1,2025-08-01 00:00:00,A,0,3,45.667
2,2025-08-01 00:00:00,A,1,2,45.000
3,2025-08-01 00:00:00,R,0,1,40.000
4,2025-08-01 00:00:00,6,1,1,37.000
...,...,...,...,...,...
386493,2025-10-10 23:50:00,L,1,1,6.000
386494,2025-10-10 23:50:00,E,1,1,5.000
386495,2025-10-10 23:50:00,J,0,1,4.000
386496,2025-10-11 00:00:00,A,0,1,39.000


In [17]:
routes = con.execute(""" SELECT DISTINCT route_id, direction_id FROM delays_recent ORDER BY route_id, direction_id""").df()

case_cols = []
for _, row in routes.iterrows():
    route = str(row["route_id"]).strip()
    direction = row["direction_id"]
    alias = f"route_{route}_{direction}".replace(" ", "_").replace("-", "_")
    expr = f"MAX(CASE WHEN route_id = '{route}' AND direction_id = {direction} THEN 1 ELSE 0 END) AS {alias}"
    case_cols.append(expr)

case_sql = ",\n    ".join(case_cols)

query = f"""
DROP TABLE IF EXISTS delays_encoded;
CREATE TABLE delays_encoded AS
SELECT
    start_interval,
    {case_sql},
    SUM(num_trips) AS total_trips,
    AVG(avg_diff_from_last_seen) AS avg_delay
FROM delays_recent
GROUP BY start_interval
ORDER BY start_interval;
"""

con.execute(query)

con.execute("SELECT * FROM delays_encoded").df()

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,route_SI_0,route_SI_1,route_SS_0,route_SS_1,route_W_0,route_W_1,route_Z_0,route_Z_1,total_trips,avg_delay
0,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,1,1,0,0,0,0,0,0,33.0,15.773821
1,2025-08-01 00:10:00,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,14.0,13.071429
2,2025-08-01 00:20:00,0,1,0,1,0,1,1,1,1,...,0,0,0,0,0,0,0,0,29.0,22.288462
3,2025-08-01 00:30:00,1,0,1,1,0,0,0,1,0,...,1,1,0,0,0,0,0,0,22.0,10.095238
4,2025-08-01 00:40:00,0,1,0,0,1,1,1,1,1,...,0,0,0,0,0,0,0,0,26.0,26.021739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10220,2025-10-10 23:20:00,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,19.0,13.611111
10221,2025-10-10 23:30:00,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,9.0,7.562500
10222,2025-10-10 23:40:00,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,16.0,9.125000
10223,2025-10-10 23:50:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4.0,5.500000


In [18]:
route_cols = con.execute("""SELECT name FROM pragma_table_info('delays_encoded') WHERE name LIKE 'r_%'""").df()["name"].tolist()

for col in route_cols:
    con.execute(
        f"""
        ALTER TABLE delays_encoded
        ALTER COLUMN {col} TYPE BOOLEAN
        USING {col}::BOOLEAN;
        """
    )

con.execute("""DESCRIBE delays_encoded""").df()

,column_name,column_type,null,key,default,extra
0,start_interval,TIMESTAMP,YES,None,None,None
1,route_1_0,BOOLEAN,YES,None,None,None
2,route_1_1,BOOLEAN,YES,None,None,None
3,route_2_0,BOOLEAN,YES,None,None,None
4,route_2_1,BOOLEAN,YES,None,None,None
...,...,...,...,...,...,...
58,route_W_1,BOOLEAN,YES,None,None,None
59,route_Z_0,BOOLEAN,YES,None,None,None
60,route_Z_1,BOOLEAN,YES,None,None,None
61,total_trips,HUGEINT,YES,None,None,None


In [19]:
con.execute(
    """
    DROP TABLE IF EXISTS traffic_recent;
    CREATE TABLE traffic_recent AS
    SELECT *
    FROM traffic_grouped
    WHERE interval BETWEEN '2025-08-01' AND '2025-10-11 23:59:59';
    """
)

con.execute("SELECT * FROM traffic_recent").df()

,interval,entries,excluded_roadway_entries,overnight,peak,monday,tuesday,wednesday,thursday,friday,...,bus,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy
0,2025-08-01 00:00:00,0.0,0.0,True,False,False,False,False,False,True,...,True,False,False,True,False,False,False,False,False,False
1,2025-08-01 00:00:00,0.0,0.0,True,False,False,False,False,False,True,...,False,True,False,False,False,False,False,True,False,False
2,2025-08-01 00:00:00,0.0,0.0,True,False,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,True
3,2025-08-01 00:00:00,0.0,0.0,True,False,False,False,False,False,True,...,False,True,False,False,False,True,False,False,False,False
4,2025-08-01 00:00:00,0.0,0.0,True,False,False,False,False,False,True,...,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393115,2025-10-04 23:50:00,277.0,53.0,True,False,False,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
393116,2025-10-04 23:50:00,343.0,0.0,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
393117,2025-10-04 23:50:00,357.0,104.0,True,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
393118,2025-10-04 23:50:00,414.0,0.0,True,False,False,False,False,False,False,...,False,False,True,False,True,False,False,False,False,False


In [20]:
con.execute(
    """
    DROP TABLE IF EXISTS trips_traffic_joined;
    CREATE TABLE trips_traffic_joined AS
    SELECT *
    FROM delays_encoded
    JOIN traffic_recent
        ON delays_encoded.start_interval = traffic_recent.interval
    ORDER BY
        delays_encoded.start_interval
    """
)

con.execute("SELECT * FROM trips_traffic_joined").df()

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,bus,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy
0,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,True,False,False,True,False,False,False,False,False,False
1,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,True,False,False,False,False,False,True,False,False
2,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,True,False,False,False,False,False,False,False,True
3,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,True,False,False,False,True,False,False,False,False
4,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393115,2025-10-04 23:50:00,True,False,False,False,False,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
393116,2025-10-04 23:50:00,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
393117,2025-10-04 23:50:00,True,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
393118,2025-10-04 23:50:00,True,False,False,False,False,False,False,False,False,...,False,False,True,False,True,False,False,False,False,False


In [21]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", None)

con.execute("""ALTER TABLE trips_traffic_joined DROP COLUMN interval""")

con.execute("""DESCRIBE trips_traffic_joined""").df()

,column_name,column_type,null,key,default,extra
0,start_interval,TIMESTAMP,YES,None,None,None
1,route_1_0,BOOLEAN,YES,None,None,None
2,route_1_1,BOOLEAN,YES,None,None,None
3,route_2_0,BOOLEAN,YES,None,None,None
4,route_2_1,BOOLEAN,YES,None,None,None
5,route_3_0,BOOLEAN,YES,None,None,None
6,route_3_1,BOOLEAN,YES,None,None,None
7,route_4_0,BOOLEAN,YES,None,None,None
8,route_4_1,BOOLEAN,YES,None,None,None
9,route_5_0,BOOLEAN,YES,None,None,None


In [22]:
con.execute(
    """
    COPY trips_traffic_joined
    TO '/content/encoded_joined_fixed.csv'
    WITH (HEADER, DELIMITER ',');
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [23]:
sample = pd.read_csv("/content/encoded_joined_fixed.csv", nrows=10)
print("Columns:", sample.columns.tolist())
display(sample.head())

Columns: ['start_interval', 'route_1_0', 'route_1_1', 'route_2_0', 'route_2_1', 'route_3_0', 'route_3_1', 'route_4_0', 'route_4_1', 'route_5_0', 'route_5_1', 'route_6_0', 'route_6_1', 'route_6X_0', 'route_6X_1', 'route_7_0', 'route_7_1', 'route_7X_0', 'route_7X_1', 'route_A_0', 'route_A_1', 'route_B_0', 'route_B_1', 'route_C_0', 'route_C_1', 'route_D_0', 'route_D_1', 'route_E_0', 'route_E_1', 'route_F_0', 'route_F_1', 'route_FS_0', 'route_FS_1', 'route_FX_0', 'route_FX_1', 'route_G_0', 'route_G_1', 'route_GS_0', 'route_GS_1', 'route_H_0', 'route_H_1', 'route_J_0', 'route_J_1', 'route_L_0', 'route_L_1', 'route_M_0', 'route_M_1', 'route_N_0', 'route_N_1', 'route_Q_0', 'route_Q_1', 'route_R_0', 'route_R_1', 'route_SI_0', 'route_SI_1', 'route_SS_0', 'route_SS_1', 'route_W_0', 'route_W_1', 'route_Z_0', 'route_Z_1', 'total_trips', 'avg_delay', 'entries', 'excluded_roadway_entries', 'overnight', 'peak', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'car_pickup_

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,route_5_1,route_6_0,route_6_1,route_6X_0,route_6X_1,route_7_0,route_7_1,route_7X_0,route_7X_1,route_A_0,route_A_1,route_B_0,route_B_1,route_C_0,route_C_1,route_D_0,route_D_1,route_E_0,route_E_1,route_F_0,route_F_1,route_FS_0,route_FS_1,route_FX_0,route_FX_1,route_G_0,route_G_1,route_GS_0,route_GS_1,route_H_0,route_H_1,route_J_0,route_J_1,route_L_0,route_L_1,route_M_0,route_M_1,route_N_0,route_N_1,route_Q_0,route_Q_1,route_R_0,route_R_1,route_SI_0,route_SI_1,route_SS_0,route_SS_1,route_W_0,route_W_1,route_Z_0,route_Z_1,total_trips,avg_delay,entries,excluded_roadway_entries,overnight,peak,monday,tuesday,wednesday,thursday,friday,saturday,sunday,car_pickup_van,single_unit_truck,multi_unit_truck,bus,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy
0,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,True,True,True,False,False,False,True,False,False,True,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,True,False,True,True,False,True,True,False,False,True,True,True,False,True,True,False,False,False,False,False,False,33,15.773821,0,0,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False
1,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,True,True,True,False,False,False,True,False,False,True,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,True,False,True,True,False,True,True,False,False,True,True,True,False,True,True,False,False,False,False,False,False,33,15.773821,0,0,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False
2,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,True,True,True,False,False,False,True,False,False,True,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,True,False,True,True,False,True,True,False,False,True,True,True,False,True,True,False,False,False,False,False,False,33,15.773821,0,0,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True
3,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,True,True,True,False,False,False,True,False,False,True,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,True,False,True,True,False,True,True,False,False,True,True,True,False,True,True,False,False,False,False,False,False,33,15.773821,0,0,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False
4,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,True,True,True,False,False,False,True,False,False,True,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,True,False,True,True,False,True,True,False,False,True,True,True,False,True,True,False,False,False,False,False,False,33,15.773821,0,0,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
